In [3]:
import pandas as pd 
from glob import glob 
root_dir = '/home/work/yuna/HPA/evaluation/scored'

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
            "Qwen/Qwen3-0.6B", 
            "Qwen/Qwen3-8B", 
            "Qwen/Qwen3-4B", 
]

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

def get_summary(dataset='mmstar'): 
    # df = df.groupby(['model', 'folder', 'condition'])['correct'].mean()

    files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/*/{dataset}*.jsonl")

    dfs= []
    for f in files: 
        try: 
            df = pd.read_json(f, lines=True)
            if 'finetuned' in f : 
                df['model'] = f.split('/')[-2].replace('fold_0', '')
            else: 
                df['model'], f = find_matching(f, [model.split('/')[-1] for model in model_names])  
            df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace('vqa 1k', '').replace(f'{dataset}', '').strip()
            dfs.append(df)
        except Exception as e: 
            print(e)
    df = pd.concat(dfs)
    print(len(files) ) 

    pt = df.pivot_table(
        index=['model'],  
        columns=['condition'], 
        values=['correct'],
        aggfunc=['mean', 'count']
    )
    pt = pt.round(4)
    pt.to_csv(f"./summary_{dataset}.csv")
    return df , pt 

In [15]:
!python /home/work/yuna/HPA/evaluation/score_humans.py --human_data_dir n20 


📊 Processing Raw Human Responses
   Session: s1
   Data dir: /home/work/yuna/HPA/data/humans/n20

📚 Loading annotations...
Length of MMStar questions: 267

Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
/home/work/yuna/HPA/data/humans/n20/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/9d6d8564_20251210_233650/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/74d408e5_20251213_134035/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/99f271ae_20251203_111155/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/ba2d2124_20251208_223639/answers.csv is incomplete, skip
✓ Loaded 12828 responses from 20 files

[2/5] Translating Korean answers...
✓ Loaded 1381 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json

🌐 Translation Status:
   Total responses: 

In [44]:
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir finetuned  
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir pretrained  

Found 178 files to score
📊 Scoring: /home/work/yuna/HPA/evaluation/results/finetuned/InternVL3_5-8B/InternVL3_5-8B_A3_vqa_15_blind_instfold_0/vqa_1k.jsonl
📂 Loading VQA annotations (this may take a moment)...
   ✓ Loaded 214354 question annotations
   ✓ Saved scored file: /home/work/yuna/HPA/evaluation/scored/finetuned/InternVL3_5-8B/InternVL3_5-8B_A3_vqa_15_blind_instfold_0/vqa_1k.jsonl

📈 Results:
   Accuracy: 0.7500 (750/1000)

   Per-category:
      is the man: 1.000 (9/9)
      does this: 1.000 (14/14)
      is the woman: 1.000 (3/3)
      is there a: 1.000 (21/21)
      is this an: 1.000 (5/5)
      what is the person: 1.000 (2/2)
      what is the color of the: 1.000 (5/5)
      are these: 1.000 (11/11)
      could: 1.000 (7/7)
      is it: 1.000 (15/15)
      has: 1.000 (2/2)
      is that a: 1.000 (4/4)
      was: 1.000 (2/2)
      are there: 1.000 (10/10)
      do you: 1.000 (4/4)
      what sport is: 1.000 (5/5)
      is he: 1.000 (3/3)
      what room is: 1.000 (2/2)
      

In [45]:
model_results = {}
for ds in ['mmstar', 'spubench', 'vqa_5k', 'vqa_1k']: 
    model_results[ds], pv = get_summary(ds) 

72
72
71
65


In [43]:
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv')
human_mc['model'] = "humans"  
human_mc.rename(columns={'mean_accuracy': 'correct'}, inplace=True) 
human_mc['condition'] = "inst blind" 
human_mc.groupby('category').agg(
    mean_correct=('correct', 'mean'),
    count=('correct', 'count')
) 
human_mc_grouped = human_mc.groupby('category').agg(
    mean_correct=('correct', 'mean'),
    count=('correct', 'count')
)
human_mc_grouped

,mean_correct,count
category,,
coarse perception,0.278877,80
fine-grained perception,0.261881,101
instance reasoning,0.282889,59
logical reasoning,0.273482,26
science & technology,0.100000,1


In [ ]:
qids = human_mc.pid.unique() 
model_mc = model_results['mmstar']
human_mc['pid'] = human_mc['pid'].astype('Int64')
model_mc['pid'] = model_mc['pid'].astype('Int64') 
print(len(qids))
mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)] , human_mc])
len(mmstar_human_comparison)
pt = mmstar_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition', 'category', 'l2_category'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
pt = pt.round(3)
pt.to_csv(f'./tables/mmstar_human_comparison.csv')
with pd.option_context('display.float_format', '{:0.3f}'.format):
    display(pt)  

247


mean  \
                                                            correct   
condition                                                             
category                                          coarse perception   
l2_category                                           image emotion   
model                                                                 
InternVL3_5-1B                                                0.667   
InternVL3_5-2B                                                0.444   
InternVL3_5-4B                                                0.778   
InternVL3_5-8B                                                0.778   
InternVL3_5-8B_A1_vqa_gt                                      0.778   
InternVL3_5-8B_A2_vqa_10_blind_inst                           0.889   
InternVL3_5-8B_A3_vqa_15_blind_inst                           0.889   
InternVL3_5-8B_A4_mmstar_15_blind_inst                        0.889   
InternVL3_5-8B_SFT_mmstar_15_blind_inst                       0.889   
Qwen3-8B-Base                                                 0.111   
Qwen3-VL-2B-Instruct                                          0.778   
Qwen3-VL-4B-Instruct                                          0.778   
Qwen3-VL-4B-Instruct_A1_vqa_gt                                0.778   
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst                     0.778   
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst                     0.778   
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst                  0.778   
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst                    0.778   
Qwen3-VL-4B-Instruct_SFT_vqa_gt                               0.778   
Qwen3-VL-8B-Instruct                                          0.778   
Qwen3-VL-8B-Instruct_A1_vqa_gt                                0.778   
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst                     0.778   
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst                     0.778   
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst                  0.778   
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst                 0.889   
Qwen3-VL-8B-Instruct_SFT_vqa_15_blind_inst                    0.889   
humans                                                          NaN   
llava-v1.6-mistral-7b-hf                                      0.556   
llava-v1.6-mistral-7b-hf_A1_vqa_gt                            0.667   
llava-v1.6-mistral-7b-hf_A2_vqa_10_blind_inst                 0.667   
llava-v1.6-mistral-7b-hf_A3_vqa_15_blind_inst                 0.667   
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst              0.667   
llava-v1.6-mistral-7b-hf_SFT_mmstar_15_blind_inst             0.667   
llava-v1.6-mistral-7b-hf_SFT_vqa_15_blind_inst                0.667   

                                                                         \
                                                                          
condition                                                                 
category                                                                  
l2_category                                       image scene and topic   
model                                                                     
InternVL3_5-1B                                                    0.531   
InternVL3_5-2B                                                    0.490   
InternVL3_5-4B                                                    0.551   
InternVL3_5-8B                                                    0.612   
InternVL3_5-8B_A1_vqa_gt                                          0.633   
InternVL3_5-8B_A2_vqa_10_blind_inst                               0.633   
InternVL3_5-8B_A3_vqa_15_blind_inst                               0.633   
InternVL3_5-8B_A4_mmstar_15_blind_inst                            0.612   
InternVL3_5-8B_SFT_mmstar_15_blind_inst                           0.653   
Qwen3-8B-Base                                                     0.184   
Qwen3-VL-2B-Instruct                                              0.571   
Qwen3-VL-4B-Instru

In [54]:
human_vqa.groupby(['model', 'condition']).mean(numeric_only=True)['correct'] 

model   condition 
humans  inst blind    0.389394
Name: correct, dtype: float64

In [40]:
human_vqa=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.csv')
# matching pids for human-model comparison 
human_vqa['model'] = "humans" 
human_vqa['condition'] = "inst blind" 
human_vqa.rename(columns={"mean_accuracy": 'correct'}, inplace=True)

qids = human_vqa.qid.unique()
print(len(qids))
model_vqa = model_results['vqa_1k'] 

vqa_human_comparison = pd.concat([model_vqa[model_vqa['question_id'].isin(qids)] , human_vqa])
vqa_human_comparison.groupby(['model', 'condition']).mean(numeric_only=True)['correct'] 

374


model                                             condition 
InternVL3_5-1B                                                  0.804813
                                                  blind         0.435829
                                                  inst blind    0.438503
InternVL3_5-2B                                                  0.820856
                                                  blind         0.455437
                                                                  ...   
llava-v1.6-mistral-7b-hf_A1_vqa_gt                inst blind    0.563280
llava-v1.6-mistral-7b-hf_A2_vqa_10_blind_inst                   0.836898
                                                  inst blind    0.475045
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst                0.859180
                                                  inst blind    0.521390
Name: correct, Length: 62, dtype: float64

In [ ]:
pt = vqa_human_comparison.pivot_table(
    index=['model'], 
    columns=['condition'], 
    values=['correct'], # , "mean_confidence" 
    aggfunc=['mean', 'count']
)
pt = pt.round(4)
pt.to_csv(f'./tables/vqa_human_comparison.csv')
pt 

mean                     \
                                                 correct                      
condition                                                  blind inst blind   
model                                                                         
InternVL3_5-1B                                    0.8048  0.4358     0.4385   
InternVL3_5-2B                                    0.8209  0.4554     0.4528   
InternVL3_5-4B                                    0.8298  0.4501     0.4724   
InternVL3_5-8B                                    0.8690  0.4715     0.4537   
InternVL3_5-8B_A1_vqa_gt                          0.8725     NaN     0.5169   
InternVL3_5-8B_A2_vqa_10_blind_inst               0.8610     NaN     0.5071   
InternVL3_5-8B_A4_mmstar_15_blind_inst            0.8636     NaN     0.4643   
Qwen3-4B                                          0.1212     NaN        NaN   
Qwen3-VL-2B-Instruct                              0.8298  0.4545     0.4617   
Qwen3-VL-4B-Instruct                              0.8592  0.4545     0.4742   
Qwen3-VL-4B-Instruct_A1_vqa_gt                    0.8815     NaN     0.4929   
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst         0.8440     NaN     0.4742   
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst         0.8494     NaN     0.4848   
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst      0.8565     NaN     0.4733   
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst        0.8476     NaN     0.4724   
Qwen3-VL-4B-Instruct_SFT_vqa_gt                   0.8868     NaN     0.5169   
Qwen3-VL-8B-Instruct                              0.8939  0.4430     0.4742   
Qwen3-VL-8B-Instruct_A1_vqa_gt                    0.8930     NaN     0.4617   
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst         0.8699     NaN     0.4608   
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst         0.8930     NaN     0.4617   
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst      0.8913     NaN     0.4679   
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst     0.8886     NaN     0.4706   
Qwen3-VL-8B-Instruct_SFT_vqa_15_blind_inst        0.8752     NaN     0.4688   
humans                                               NaN     NaN     0.3894   
llava-v1.6-mistral-7b-hf                          0.8627  0.4510     0.4670   
llava-v1.6-mistral-7b-hf_A1_vqa_gt                0.8770     NaN     0.5633   
llava-v1.6-mistral-7b-hf_A2_vqa_10_blind_inst     0.8369     NaN     0.4750   
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst  0.8592     NaN     0.5214   

                                                   count                    
                                                 correct                    
condition                                                 blind inst blind  
model                                                                       
InternVL3_5-1B                                     374.0  374.0      374.0  
InternVL3_5-2B                                     374.0  374.0      374.0  
InternVL3_5-4B                                     374.0  374.0      374.0  
InternVL3_5-8B                                     374.0  374.0      374.0  
InternVL3_5-8B_A1_vqa_gt                           374.0    NaN      374.0  
InternVL3_5-8B_A2_vqa_10_blind_inst                374.0    NaN      374.0  
InternVL3_5-8B_A4_mmstar_15_blind_inst             374.0    NaN      374.0  
Qwen3-4B                                           374.0    NaN        NaN  
Qwen3-VL-2B-Instruct                               374.0  374.0      374.0  
Qwen3-VL-4B-Instruct                               374.0  374.0      374.0  
Qwen3-VL-4B-Instruct_A1_vqa_gt                     374.0    NaN      374.0  
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst          374.0    NaN      374.0  
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst          374.0    NaN      374.0  
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst       374.0    NaN      374.0  
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst         374.0    NaN      374.0  
Qwen3-VL-4B-Instruct_SFT_vqa_gt                    374.0    

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl